In [1]:
from FCN_model import FCN
import torch
import torch.optim as optim
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np
import pandas as pd
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    torch.cuda.manual_seed_all(42)
    torch.backends.cudnn.deterministic = True

In [2]:
origin_FCN = FCN(in_channels = 1, num_classes=2).to("cuda")

In [3]:
origin_FCN

FCN(
  (layers): Sequential(
    (0): Conv1d(1, 128, kernel_size=(8,), stride=(1,), padding=same)
    (1): BatchNorm1d(128, eps=0.001, momentum=0.01, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv1d(128, 256, kernel_size=(5,), stride=(1,), padding=same)
    (4): BatchNorm1d(256, eps=0.001, momentum=0.01, affine=True, bias=True, track_running_stats=True)
    (5): ReLU()
    (6): Conv1d(256, 128, kernel_size=(3,), stride=(1,), padding=same)
    (7): BatchNorm1d(128, eps=0.001, momentum=0.01, affine=True, bias=True, track_running_stats=True)
    (8): ReLU()
    (9): AdaptiveAvgPool1d(output_size=1)
    (10): Flatten(start_dim=1, end_dim=-1)
    (11): Linear(in_features=128, out_features=2, bias=True)
  )
)

In [4]:
epochs = 2000
optimizer = torch.optim.Adam(origin_FCN.parameters(), lr=0.001)

criterion = nn.CrossEntropyLoss()

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=50,
    min_lr=0.0001
)

#### Test with original dataset without z-score again

In [5]:
FordA_train = np.load("FordA_processed/FordA_TRAIN.npy")
FordA_test = np.load("FordA_processed/FordA_TEST.npy")

In [6]:
FordA_X_train = np.expand_dims(FordA_train[:, 1:], axis=1)
FordA_y_train = FordA_train[:, 0]

In [7]:
print(FordA_X_train.shape)
print(FordA_y_train.shape)

(3601, 1, 500)
(3601,)


In [8]:
FordA_y_train = (FordA_y_train == 1).astype(np.int64)


In [9]:
FordA_X_test = np.expand_dims(FordA_test[:, 1:], axis=1)
FordA_y_test = FordA_test[:, 0]
FordA_y_test = (FordA_y_test == 1).astype(np.int64)

In [10]:
FordA_X_train = torch.tensor(FordA_X_train, dtype=torch.float32)
FordA_y_train = torch.tensor(FordA_y_train, dtype=torch.long)

In [11]:
FordA_X_test = torch.tensor(FordA_X_test, dtype=torch.float32)
FordA_y_test = torch.tensor(FordA_y_test, dtype=torch.long)

In [12]:
dataset = TensorDataset(FordA_X_train, FordA_y_train)

train_loader = DataLoader(
    dataset,
    batch_size=360, # calculated as FordA_X_train.shape[0] / 10, according to tensorflow version
    shuffle=True
)

#### Training experiment 1

In [13]:
metrics = {
    "epoch": [],
    "loss": [],
    "accuracy": [],
    "precision": [],
    "recall": [],
    "f1": []
}

In [14]:
min_loss = 0.2
for epoch in range(epochs):
    metrics["epoch"].append(epoch)
    origin_FCN.train()
    running_loss = 0.0
    total_samples = 0
    all_predictions = []
    all_labels = []

    for X, y in train_loader:
        X = X.to("cuda")
        y = y.to("cuda")

        optimizer.zero_grad()
        # print(batch_features.shape)
        # Forward pass
        outputs = origin_FCN(X)
        loss = criterion(outputs, y)

        # print(outputs)
        # print(loss)

         # Backward pass and optimization
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * X.size(0)
        total_samples += X.size(0)

        predictions = outputs.argmax(dim=1)
        all_predictions.extend(
            predictions.detach().cpu().numpy()
        )
        all_labels.extend(
            y.cpu().numpy()
        )
    epoch_loss = running_loss / total_samples

    accuracy = accuracy_score(all_labels, all_predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
                                                        all_labels,
                                                        all_predictions,
                                                        average="macro",
                                                        zero_division=0
                                                    )
    # Save metrics
    if epoch_loss <= min_loss:
        torch.save({
                "model_state_dict": origin_FCN.state_dict(),
                "metrics": metrics,
                "optimizer_state_dict": optimizer.state_dict()
            }, f"trained_model/fcn_model_min_loss_epoch{epoch}_loss{epoch_loss: .5f}.pth")
        # torch.save(origin_FCN.state_dict(), f"fcn_model_highest_accuracy_epoch{epoch}.pth")
        min_loss = epoch_loss
    metrics["loss"].append(epoch_loss)
    metrics["accuracy"].append(accuracy)
    metrics["precision"].append(precision)
    metrics["recall"].append(recall)
    metrics["f1"].append(f1)

    print(
            f"Epoch {epoch+1}/{epochs} | "
            f"Loss: {epoch_loss:.4f} | "
            f"Accuracy: {accuracy:.4f} | "
            f"Precision: {precision:.4f} | "
            f"Recall: {recall:.4f} | "
            f"F1: {f1:.4f}"
    )
    #     break
    # break
metrics_df = pd.DataFrame(metrics)
metrics_df.to_csv("FCN_original_metrics.csv")

c:\Users\96561\miniconda3\envs\UCR-pytorch\Lib\site-packages\torch\nn\modules\conv.py:380: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\native\Convolution.cpp:1105.)
  return F.conv1d(


Epoch 1/2000 | Loss: 0.5498 | Accuracy: 0.7148 | Precision: 0.7560 | Recall: 0.7197 | F1: 0.7057
Epoch 2/2000 | Loss: 0.4445 | Accuracy: 0.7998 | Precision: 0.7999 | Recall: 0.7993 | F1: 0.7995
Epoch 3/2000 | Loss: 0.3960 | Accuracy: 0.8364 | Precision: 0.8364 | Recall: 0.8365 | F1: 0.8364
Epoch 4/2000 | Loss: 0.3818 | Accuracy: 0.8387 | Precision: 0.8398 | Recall: 0.8394 | F1: 0.8386
Epoch 5/2000 | Loss: 0.3715 | Accuracy: 0.8406 | Precision: 0.8412 | Recall: 0.8411 | F1: 0.8406
Epoch 6/2000 | Loss: 0.3460 | Accuracy: 0.8648 | Precision: 0.8657 | Recall: 0.8641 | F1: 0.8645
Epoch 7/2000 | Loss: 0.3425 | Accuracy: 0.8748 | Precision: 0.8762 | Recall: 0.8740 | F1: 0.8744
Epoch 8/2000 | Loss: 0.3175 | Accuracy: 0.8814 | Precision: 0.8821 | Recall: 0.8809 | F1: 0.8812
Epoch 9/2000 | Loss: 0.3140 | Accuracy: 0.8875 | Precision: 0.8884 | Recall: 0.8869 | F1: 0.8873
Epoch 10/2000 | Loss: 0.3089 | Accuracy: 0.8781 | Precision: 0.8781 | Recall: 0.8783 | F1: 0.8781
Epoch 11/2000 | Loss: 0.2940 

In [16]:
metrics_df.sort_values(by="loss")

,epoch,loss,accuracy,precision,recall,f1
1905,1905,0.111185,0.963621,0.963556,0.963661,0.963602
1887,1887,0.113149,0.963066,0.963208,0.962909,0.963031
1891,1891,0.113252,0.962510,0.962520,0.962451,0.962484
1890,1890,0.113446,0.961400,0.961340,0.961424,0.961379
1997,1997,0.113955,0.960844,0.960868,0.960770,0.960815
...,...,...,...,...,...,...
4,4,0.371503,0.840600,0.841154,0.841144,0.840600
3,3,0.381829,0.838656,0.839829,0.839417,0.838643
2,2,0.395996,0.836434,0.836354,0.836547,0.836389
1,1,0.444533,0.799778,0.799920,0.799320,0.799483


#### Inference on test

In [17]:
origin_FCN = FCN(in_channels = 1, num_classes=2).to("cuda")

In [18]:
test_dataset = TensorDataset(FordA_X_test, FordA_y_test)

test_loader = DataLoader(
    test_dataset,
    batch_size=360, # calculated as FordA_X_train.shape[0] / 10, according to tensorflow version
    shuffle=False
)

In [19]:
state_dict = torch.load(
    "fcn_model_highest_accuracy_epoch1905.pth",
    map_location="cuda",
    weights_only=True
)

In [20]:
origin_FCN.load_state_dict(state_dict["model_state_dict"])
origin_FCN = origin_FCN.to("cuda")
origin_FCN.eval()

FCN(
  (layers): Sequential(
    (0): Conv1d(1, 128, kernel_size=(8,), stride=(1,), padding=same)
    (1): BatchNorm1d(128, eps=0.001, momentum=0.01, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv1d(128, 256, kernel_size=(5,), stride=(1,), padding=same)
    (4): BatchNorm1d(256, eps=0.001, momentum=0.01, affine=True, bias=True, track_running_stats=True)
    (5): ReLU()
    (6): Conv1d(256, 128, kernel_size=(3,), stride=(1,), padding=same)
    (7): BatchNorm1d(128, eps=0.001, momentum=0.01, affine=True, bias=True, track_running_stats=True)
    (8): ReLU()
    (9): AdaptiveAvgPool1d(output_size=1)
    (10): Flatten(start_dim=1, end_dim=-1)
    (11): Linear(in_features=128, out_features=2, bias=True)
  )
)

In [21]:
running_loss = 0.0
total_samples = 0
all_predictions = []
all_labels = []

test_metrics = {
    "loss": [],
    "accuracy": [],
    "precision": [],
    "recall": [],
    "f1": []
}

criterion = torch.nn.CrossEntropyLoss()

with torch.no_grad():
    for X, y in test_loader:
        X = X.to("cuda")
        y = y.to("cuda")

        # Forward pass
        outputs = origin_FCN(X)
        loss = criterion(outputs, y)

        # Predictions
        predictions = torch.argmax(outputs, dim=1)

        # Accumulate loss
        running_loss += loss.item() * X.size(0)
        total_samples += X.size(0)

        # Collect predictions and labels
        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(y.cpu().numpy())
# Compute evaluation metrics
test_loss = running_loss / total_samples
accuracy = accuracy_score(all_labels, all_predictions)

precision, recall, f1, _ = precision_recall_fscore_support(
    all_labels,
    all_predictions,
    average="macro",
    zero_division=0
)

# Store metrics
test_metrics = {
    "loss": test_loss,
    "accuracy": accuracy,
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "predictions": all_predictions,
    "labels": all_labels
}

print(
    f"Test | "
    f"Loss: {test_loss:.4f} | "
    f"Accuracy: {accuracy:.4f} | "
    f"Precision: {precision:.4f} | "
    f"Recall: {recall:.4f} | "
    f"F1: {f1:.4f}"
)

Test | Loss: 0.1992 | Accuracy: 0.9273 | Precision: 0.9276 | Recall: 0.9279 | F1: 0.9273
